# 203 — Predict Sentinel-2

Downloads S2 tiles, predicts lupine probability, saves binary + prob tiles.
Run after `202_train_sat.ipynb`.

In [ ]:
from config_s2 import *
from helpers_s2 import *
import time, gc
import numpy as np
import joblib

print(f"Config loaded ✓  |  Mode: {MODE}  |  Thresholds: {PIXEL_THRESHOLDS}")
check_required_files_s2()

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
rf = joblib.load(MODEL_PATH_S2)
rf.n_jobs = -1
print(f"Model: {MODEL_NAME_S2}  |  Trees: {rf.n_estimators}  |  Features: {rf.n_features_in_}")

In [ ]:
# ── Build tile list ───────────────────────────────────────────────────────────
print("── Building tile list ──────────────────────────────────────")

grid = gpd.read_file(S2_GRID_PATH)

if MODE == "region":
    gdf_regions = gpd.read_file(REGION_PATH)
    region_row  = gdf_regions[gdf_regions[REGION_ID_COL] == REGION_ID]
    region_geom = region_row.to_crs(grid.crs).geometry.values[0]
    work_grid   = grid[grid.intersects(region_geom)].copy()
    print(f"  Mode: region {REGION_ID}  |  Tiles: {len(work_grid)}")
else:
    work_grid = grid.copy()
    print(f"  Mode: all  |  Total: {len(work_grid)}")

url_map    = {row["full_name"]: row["url"] for _, row in work_grid.iterrows()}
all_fnames = sorted(url_map.keys())

# Resume
ref_thresh      = PIXEL_THRESHOLDS[0]
existing_binary = set(os.listdir(s2_binary_tile_dir(ref_thresh)))
remaining       = [f for f in all_fnames
                   if f.replace(".tif", f"_binary_t{ref_thresh}.tif") not in existing_binary]
print(f"  Total: {len(all_fnames)}  Done: {len(all_fnames)-len(remaining)}  Remaining: {len(remaining)}")

In [ ]:
# ── Download → predict → save ────────────────────────────────────────────────
print("── Processing tiles ────────────────────────────────────────")

skipped = 0
t0      = time.time()

for i, fname in enumerate(remaining):
    url             = url_map[fname]
    local_path      = os.path.join(TMP_DIR, fname)
    remaining_count = len(remaining) - i - 1

    if not os.path.exists(local_path):
        ok = safe_download(url, local_path)
        if not ok:
            skipped += 1; continue

    try:
        with rasterio.open(local_path) as src:
            img     = src.read()
            profile = src.profile.copy()
    except Exception as e:
        print(f"  [!] Read failed {fname}: {e}")
        skipped += 1
        try: os.remove(local_path)
        except: pass
        continue

    # Predict
    prob      = rf.predict_proba(engineer_features_s2(img, NODATA_VAL, S2_SCALE))[:, 1].astype(np.float32)
    prob_mask = prob.reshape(img.shape[1], img.shape[2])
    del prob; gc.collect()

    # Save prob tile
    prob_profile = profile.copy()
    prob_profile.update(count=1, dtype="float32", compress="lzw",
                        tiled=True, blockxsize=256, blockysize=256, nodata=None)
    with rasterio.open(os.path.join(s2_prob_tile_dir(), fname), "w", **prob_profile) as dst:
        dst.write(prob_mask[np.newaxis, :, :])

    # Save binary tiles
    for thresh in PIXEL_THRESHOLDS:
        binary      = (prob_mask >= thresh).astype(np.uint8)
        bin_fname   = fname.replace(".tif", f"_binary_t{thresh}.tif")
        bin_profile = profile.copy()
        bin_profile.update(count=1, dtype="uint8", compress="lzw",
                           tiled=True, blockxsize=256, blockysize=256, nodata=None)
        with rasterio.open(os.path.join(s2_binary_tile_dir(thresh), bin_fname), "w", **bin_profile) as dst:
            dst.write(binary[np.newaxis, :, :])
        del binary

    del img, prob_mask; gc.collect()

    if DELETE_AFTER:
        try: os.remove(local_path)
        except: pass

    elapsed = time.time() - t0
    done    = i + 1 - skipped
    rate    = done / elapsed if elapsed > 0 else 0
    eta     = remaining_count / rate if rate > 0 else 0

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == len(remaining):
        print(f"\n  ── Checkpoint [{i+1}/{len(remaining)}] ──────────────")
        print(f"     Elapsed: {elapsed/3600:.1f}h  |  ETA: {eta/3600:.1f}h\n")
    else:
        print(f"  [{i+1}/{len(remaining)}] {fname} — {remaining_count} left — ETA {eta/60:.1f}min")

print(f"\n  Done. Skipped: {skipped}  |  Time: {(time.time()-t0)/3600:.1f}h")